# 01 - Data Extraction

Compiled version of the extraction notebook.

Design choices:
- Keep the core project universe from the other branches: Gold, Brent, DXY, VIX, and US10Y.
- Download Copper (`HG=F`) as a robustness-extension asset, not as part of the primary risk book yet.
- Use explicit ticker-to-asset mapping instead of positional column renaming.
- Save raw yfinance output and a clean close-price panel for Step 02.


## Reader Orientation

This notebook starts the nowcasting workflow. It does not try to prove the dashboard yet. Its job is to create a clean market data universe where Gold can later be evaluated as a real-time instability indicator. Brent is the core commodity book exposure; Copper is kept available only for robustness.

In [20]:
from pathlib import Path

import pandas as pd
import numpy as np
import yfinance as yf

ROOT = Path.cwd()
RAW_DIR = ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)


In [21]:
CORE_TICKERS = {
    "Gold": "GC=F",          # Gold futures; primary alarm asset
    "Brent": "BZ=F",         # Brent crude futures; primary benchmark risk-book exposure
    "DXY": "DX-Y.NYB",       # US Dollar Index
    "VIX": "^VIX",           # CBOE Volatility Index
    "US10Y": "^TNX",         # 10Y Treasury yield, quoted as yield x 10
}

ROBUSTNESS_TICKERS = {
    "Copper": "HG=F",        # Copper futures; robustness-extension commodity exposure
}

TICKERS = {**CORE_TICKERS, **ROBUSTNESS_TICKERS}

START_DATE = "2010-01-01"
END_DATE = None
PRICE_FIELD = "Close"

TICKER_TO_ASSET = {ticker: asset for asset, ticker in TICKERS.items()}
TICKERS


{'Gold': 'GC=F',
 'Brent': 'BZ=F',
 'DXY': 'DX-Y.NYB',
 'VIX': '^VIX',
 'US10Y': '^TNX',
 'Copper': 'HG=F'}

### Why This Asset Set

Gold is the alarm asset, Brent is the primary commodity risk-book exposure, and DXY/VIX/US10Y are conditioning variables used to interpret what gold is reacting to. Copper is downloaded here only so it is available later for a diversified commodity-book robustness check. Keeping it outside the core book avoids changing the main hypothesis before the Brent-based test is evaluated.

In [22]:
raw_data = yf.download(
    tickers=list(TICKERS.values()),
    start=START_DATE,
    end=END_DATE,
    auto_adjust=False,
    group_by="column",
    threads=True,
)

raw_data.tail()


[*********************100%***********************]  6 of 6 completed


Price       Adj Close                                                    \
Ticker           BZ=F   DX-Y.NYB         GC=F    HG=F   ^TNX       ^VIX   
Date                                                                      
2026-05-27  94.290001  99.209999  4447.500000  6.3050  4.481  16.290001   
2026-05-28  93.709999  99.019997  4499.299805  6.3960  4.455  15.740000   
2026-05-29  92.050003  98.910004  4560.500000  6.3595  4.453  15.320000   
2026-06-01  94.980003  99.199997  4475.200195  6.5240  4.475  16.049999   
2026-06-02  93.669998  99.078003  4557.299805  6.6480    NaN  16.160000   

Price           Close                                  ...         Open  \
Ticker           BZ=F   DX-Y.NYB         GC=F    HG=F  ...         GC=F   
Date                                                   ...                
2026-05-27  94.290001  99.209999  4447.500000  6.3050  ...  4439.700195   
2026-05-28  93.709999  99.019997  4499.299805  6.3960  ...  4453.600098   
2026-05-29  92.050003  98.910004  4560.500000  6.3595  ...  4494.000000   
2026-06-01  94.980003  99.199997  4475.200195  6.5240  ...  4523.500000   
2026-06-02  93.669998  99.078003  4557.299805  6.6480  ...  4515.799805   

Price                                  Volume                                  \
Ticker        HG=F   ^TNX       ^VIX     BZ=F DX-Y.NYB     GC=F     HG=F ^TNX   
Date                                                                            
2026-05-27  6.4175  4.471  17.010000  20553.0      0.0  81062.0   2474.0  0.0   
2026-05-28  6.3000  4.508  16.760000   9881.0      0.0  16427.0   2596.0  0.0   
2026-05-29  6.3920  4.441  15.810000  39308.0      0.0   1883.0   1317.0  0.0   
2026-06-01  6.5440  4.457  15.880000  39308.0      0.0   1883.0   1317.0  0.0   
2026-06-02  6.5645    NaN  16.280001  12160.0      0.0  50556.0  16999.0  NaN   

Price            
Ticker     ^VIX  
Date             
2026-05-27  0.0  
2026-05-28  0.0  
2026-05-29  0.0  
2026-06-01  0.0  
2026-06-02  0.0  

[5 rows x 36 columns]

In [23]:
def extract_price_panel(raw: pd.DataFrame, price_field: str = "Close") -> pd.DataFrame:
    if isinstance(raw.columns, pd.MultiIndex):
        fields = raw.columns.get_level_values(0)
        if price_field in fields:
            prices = raw[price_field].copy()
        elif "Adj Close" in fields:
            prices = raw["Adj Close"].copy()
        else:
            raise ValueError(f"Could not find {price_field!r} or 'Adj Close' in yfinance output")
    else:
        prices = raw.copy()

    prices = prices.rename(columns=TICKER_TO_ASSET)
    prices = prices[list(TICKERS.keys())]
    prices.index = pd.to_datetime(prices.index)
    return prices.sort_index()


prices = extract_price_panel(raw_data, PRICE_FIELD)
prices.tail()


Ticker,Gold,Brent,DXY,VIX,US10Y,Copper
Date,,,,,,
2026-05-27,4447.500000,94.290001,99.209999,16.290001,4.481,6.3050
2026-05-28,4499.299805,93.709999,99.019997,15.740000,4.455,6.3960
2026-05-29,4560.500000,92.050003,98.910004,15.320000,4.453,6.3595
2026-06-01,4475.200195,94.980003,99.199997,16.049999,4.475,6.5240
2026-06-02,4557.299805,93.669998,99.078003,16.160000,NaN,6.6480


In [24]:
raw_path = RAW_DIR / "yfinance_raw_data.parquet"
prices_path = RAW_DIR / "prices_close_raw.parquet"

raw_data.to_parquet(raw_path)
prices.to_parquet(prices_path)

print("Saved raw data to:", raw_path)
print("Saved close-price panel to:", prices_path)
print("Core assets:", list(CORE_TICKERS))
print("Robustness assets:", list(ROBUSTNESS_TICKERS))
print("Shape:", prices.shape)
print("Date range:", prices.index.min(), "to", prices.index.max())
print("Missing values:")
print(prices.isna().sum())


Saved raw data to: C:\Users\sohwe\Desktop\SMU\MQF\Commodities Risk Management\qf637\data\raw\yfinance_raw_data.parquet
Saved close-price panel to: C:\Users\sohwe\Desktop\SMU\MQF\Commodities Risk Management\qf637\data\raw\prices_close_raw.parquet
Core assets: ['Gold', 'Brent', 'DXY', 'VIX', 'US10Y']
Robustness assets: ['Copper']
Shape: (4134, 6)
Date range: 2010-01-04 00:00:00 to 2026-06-02 00:00:00
Missing values:
Ticker
Gold       7
Brent     37
DXY        5
VIX        5
US10Y      9
Copper     6
dtype: int64


### Result Comment And Significance

In the latest run, missing observations were small across the downloaded assets: Brent had the most missing rows, but still less than 1% of the sample. That supports using daily yfinance data for a prototype, but it also motivates Step 02's strict alignment choice. The missing rows are likely calendar/holiday mismatches rather than a reason to forward-fill prices into the signal.